# 교시별 실습 Notebook

d3-p01-02

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더를 확인했습니다.")

# 요약

제공 삼성전자 주가 CSV로 시작하고 yfinance 수집 방법을 익힌 뒤, 다음 거래일 종가를 예측할 특성과 날짜 분리를 만듭니다.

## 3일차 1~2교시 — 주가 수집과 미래 예측용 특성

2026년 10월 8일(목) · 3일차 1~2교시 · 1차 초안. 공식 안내문은 일차별 단원과 총시간을 정하며, 이 페이지의 교시별 묶음은 수업 진행을 위한 제안입니다. 같은 실습을 이어가는 교시는 한 페이지로 묶었습니다.

## 시작 전에

새 Notebook `day3_stock_regression`을 만듭니다. 기본 실습은 pandas와 제공 CSV만으로 실행합니다. 공식 수집 실습에는 yfinance를 사용합니다. 접속 실패 시 제공 CSV로 분석을 이어갑니다. 예측 시점은 오늘 장 마감 후, 정답은 다음 거래일 종가입니다. 3~4교시에 같은 Notebook을 이어 씁니다.

코드 블록 하나를 Notebook의 코드 셀 하나에 붙여 넣고 위에서 아래로 Shift+Enter로 실행합니다. 앞 셀을 실행하지 않고 뒤 셀부터 실행하면 변수를 찾지 못할 수 있습니다. 파일 경로는 수업 폴더 `kpc-lec`를 기준으로 합니다. 현재 위치가 다르면 파일 경로를 실제 파일 위치로 바꿉니다.

## 교시별 진행 제안

- 1교시: 예측 질문과 시점 정의 → CSV 읽기 → 날짜 정렬·중복·결측 확인 → yfinance 수집 시연·직접 실행·실패 원인 기록.
- 2교시: 수익률·이동평균·lag 특성 → 다음 거래일 정답 → 날짜 순서로 분리 → 경계에서 미래 정답이 섞이지 않는지 확인.

## 1. yfinance 또는 FinanceDataReader를 활용한 주가 데이터 수집

공식 항목의 두 수집 도구 중 yfinance를 사용합니다. 1교시에서 수집 코드·기간·조정 옵션을 설명하고 강사 시연 후 직접 실행합니다. `prices_live = load_prices(use_live=True)`를 별도 셀에서 실행해 행 수와 기간을 확인합니다. 수집 결과는 `prices_live.to_csv("practice/prices-live.csv")`로 저장합니다. 비교 실습은 검증한 제공 CSV를 공통 자료로 사용하므로 뒤의 기본 코드는 `use_live=False`로 진행합니다. 기본 코드는 제공 파일을 읽으므로 네트워크 없이 실행할 수 있습니다. `use_live=True`는 지정된 과거 기간을 새로 수집하는 현장 실습입니다.

In [ ]:
from pathlib import Path
import pandas as pd

def load_prices(use_live=False):
    if use_live:
        import yfinance as yf
        result = yf.download("005930.KS", start="2025-01-01", end="2026-01-01",
                             auto_adjust=True, multi_level_index=False, progress=False)
        if result is None or result.empty:
            raise RuntimeError("주가 수집 결과가 비었습니다. 제공 CSV로 실습하세요.")
        result.index = pd.to_datetime(result.index).tz_localize(None)
    else:
        path = Path("raw/기존강사 수업자료/_ko_stock/stock_data/005930.csv")
        result = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")
    result = result.sort_index()
    result = result.loc[~result.index.duplicated(keep="last")].copy()
    result["Close"] = pd.to_numeric(result["Close"], errors="raise")
    assert result.index.is_monotonic_increasing
    assert (result["Close"].dropna() > 0).all()
    return result

prices = load_prices(use_live=False)
Path("practice").mkdir(exist_ok=True)
print("행·열:", prices.shape)
print("기간:", prices.index.min().date(), "~", prices.index.max().date())
print(prices[["Close"]].head())

제공 CSV는 Date를 인덱스로 옮긴 후 401행·6열입니다. 수집 코드의 `end`는 제외되는 끝 날짜입니다. `auto_adjust=True`는 조정된 가격을 사용하며, 제공 CSV의 Close와 기간·가격 처리 기준이 다를 수 있습니다. 같은 실험을 비교할 때는 사용한 파일·기간·조정 옵션을 기록합니다. [yfinance download 공식 문서](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)를 참고합니다. 네트워크 수집은 외부 응답에 따라 실패할 수 있으므로 제공 CSV가 기본 실습 자료입니다.

**직접 실습:** 라이브 수집의 성공 여부·수집 시각·도구·기간·조정 옵션을 기록합니다. 제공 CSV와 라이브 결과는 파일을 분리합니다. 아래 401행·396행·315행·80행은 제공 CSV에 대한 값이며 라이브 자료의 고정 정답이 아닙니다. 처음·마지막 날짜와 결측 종가 개수를 확인합니다. `use_live=True`는 패키지 설치와 통신 상태를 확인한 뒤 별도 셀에서 실행합니다.

## 2. 종가, 수익률, 이동평균, lag feature 생성

시점 t의 종가·수익률·5거래일 이동평균·이전 거래일 종가를 입력으로 사용합니다. 정답은 t의 다음 거래일 종가입니다. 주말·휴장일이 있어 “다음 행”이 항상 다음 달력 날짜는 아닙니다.

In [ ]:
frame = pd.DataFrame(index=prices.index)
frame["close"] = prices["Close"]
frame["return_1d"] = prices["Close"].pct_change(fill_method=None)
frame["ma_5"] = prices["Close"].rolling(5).mean()
frame["lag_close_1"] = prices["Close"].shift(1)
frame["target_next_close"] = prices["Close"].shift(-1)
frame["target_date"] = pd.Series(prices.index, index=prices.index).shift(-1)
frame = frame.dropna().copy()
print(frame.head().round(4))
print("학습 가능한 행:", len(frame))
feature_columns = ["close", "return_1d", "ma_5", "lag_close_1"]
X = frame[feature_columns]
y = frame["target_next_close"]
split = int(len(frame) * 0.8)
test_start = frame.index[split]
# 훈련 정답 날짜도 테스트 입력의 첫 날짜보다 앞서도록 경계 행을 제외합니다.
train_mask = (frame.index < test_start) & (frame["target_date"] < test_start)
test_mask = frame.index >= test_start
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
assert frame.loc[train_mask, "target_date"].max() < X_test.index.min()
print("훈련·테스트:", len(X_train), len(X_test))
print("훈련 마지막 정답 날짜:", frame.loc[train_mask, "target_date"].max().date())
print("테스트 첫 입력 날짜:", X_test.index.min().date())

첫 4행은 이동평균, 마지막 1행은 다음 날 정답을 만들 수 없어 빠집니다. 396행 중 훈련 315행, 경계 제외 1행, 테스트 80행입니다. 이 구조는 오늘의 거래를 모두 관찰한 뒤 다음 거래일을 예측하는 상황에 맞습니다. 장 시작 전 예측이라면 오늘 종가와 오늘 수익률도 아직 모르는 정보이므로 입력을 다시 설계해야 합니다.

**직접 실습:** 이동평균을 10거래일로 바꾸고 유효 행 수를 확인합니다. `shift(1)`과 `shift(-1)`이 각각 과거·미래 중 어느 값을 옮기는지 설명합니다. 테스트 구간을 무작위로 섞지 않습니다.

## 직접 해보기와 정답

정답을 미래에서 가져오는 것과 입력에 미래 정보를 넣는 것을 구분합니다. 아래 셀로 시점 관계를 확인합니다.

In [ ]:
print(frame[["close", "lag_close_1", "target_next_close", "target_date"]].head(6))
print("종가 결측 수:", prices["Close"].isna().sum())
print("전체 / 훈련 / 경계 제외 / 테스트:", len(frame), len(X_train), len(frame) - len(X_train) - len(X_test), len(X_test))

이동평균을 10거래일로 바꿀 때는 원본 prices에서 frame을 만드는 셀을 수정하여 다시 실행합니다. `ma_5`를 `ma_10`으로 바꾸고 feature_columns도 함께 수정해야 합니다.

<details>
<summary>정답</summary>

shift(1)은 이전 거래일 종가를 현재 행으로 가져오고, shift(-1)은 다음 거래일 종가를 정답으로 가져옵니다. 5거래일 평균은 첫 4행, 10거래일 평균은 첫 9행에서 계산되지 않습니다. 제공 파일 401행에서 다른 결측이 없으면 10거래일 평균과 다음 거래일 정답으로 남는 행은 391개입니다. 입력에는 과거·현재 정보만 있고 미래 종가는 y에만 있어야 합니다.

</details>

## 흔한 오류와 해결

| 증상 | 해결 |
| --- | --- |
| CSV 경로 FileNotFoundError | `_ko_stock/stock_data/005930.csv`의 실제 위치를 확인합니다. |
| 수집 결과가 비었음 | 기본 use_live=False로 제공 CSV를 사용합니다. |
| yfinance의 multi_level_index 인자 오류 | 선택 수집 패키지 버전을 확인합니다. 기본 CSV 실습은 계속 진행할 수 있습니다. |
| 특성 변경 뒤 KeyError | frame 열 이름과 feature_columns를 함께 수정합니다. |
| 오늘과 내일 날짜가 하루 이상 차이 남 | 주말·휴장일은 거래가 없으므로 다음 거래일은 다음 달력 날짜와 다를 수 있습니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 파일·기간·가격 조정 기준을 기록했습니다.
- [ ] 현재 정보와 미래 정답을 구분했습니다.
- [ ] 훈련 정답 마지막 날짜가 테스트 첫 입력보다 앞선 것을 확인했습니다.
- [ ] 제공 자료 기준 유효 396행·훈련 315행·경계 제외 1행·테스트 80행을 확인했습니다.

## 기준 자료

공식 안내문 3쪽의 「주가 데이터 기반 회귀 분석 프로젝트」 4시간 중 앞 2교시를 기준으로 구성했습니다. 기존 단원 자료의 실습 코드를 교시별로 나눈 초안입니다. 실행 결과의 행 수는 제공 파일을 기준으로 하며, 라이브 수집 자료나 라이브러리 버전이 달라지면 값이 달라질 수 있습니다.

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 7일 7~8교시 — 신용카드 부도 예측과 평가](surl:14901) · [다음: 10월 8일 3~4교시 — 주가 회귀와 기준 모델 비교](surl:14903)
<!-- period-navigation:end -->